In [ ]:
# %pip install -q \
# transformers>=4.44.0 \
# datasets>=2.20.0 \
# peft>=0.12.0 \
# trl>=0.9.6 \
# accelerate>=0.34.0 \
# bitsandbytes \
# evaluate \
# scikit-learn

In [ ]:
from pathlib import Path

import torch
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    TaskType,
    PeftModel
)

from trl import SFTTrainer

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

OUTPUT_DIR = "./qwen_lora"

MAX_SAMPLES = 5000

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

In [ ]:
dataset = load_dataset(
    "claritystorm/cfpb-consumer-complaints"
)

dataset

In [ ]:
df = dataset["train"].to_pandas()

df.head()

In [ ]:
sample = dataset["train"][0]

for k in sample.keys():
    print(k)

In [ ]:
dataset_train = dataset["train"]

dataset_train = dataset_train.filter(
    lambda x:
    x["consumer_narrative"] is not None
)

dataset_train = dataset_train.select(
    range(
        min(
            MAX_SAMPLES,
            len(dataset_train)
        )
    )
)

dataset_train

In [ ]:
def estimate_urgency(text):
    text = text.lower()

    high_keywords = [
        "fraude",
        "golpe",
        "roubo",
        "ameaça",
        "cobrança indevida",
        "conta bloqueada",
        "cartão clonado"
    ]

    medium_keywords = [
        "atraso",
        "cancelamento",
        "erro",
        "problema",
        "reembolso"
    ]

    if any(word in text for word in high_keywords):
        return "Alta"

    if any(word in text for word in medium_keywords):
        return "Média"

    return "Baixa"

In [ ]:
def create_example(example):

    complaint = str(example["consumer_narrative"])

    urgency = estimate_urgency(complaint)

    user_prompt = f"""
Analise a reclamação abaixo.

Retorne:

1. Categoria
2. Produto
3. Urgência
4. Problema Principal
5. Resposta Inicial

Reclamação:

{complaint}
"""

    assistant_answer = f"""
Categoria: {example['product_normalised']}

Produto: {example['product']}

Urgência: {urgency}

Problema Principal:
{example['issue']}

Resposta Inicial:
Recebemos sua reclamação referente ao produto '{example['product']}'.
Nossa equipe analisará o caso e retornará o mais breve possível.
"""

    return {
        "messages": [
            {
                "role": "system",
                "content": "Você é um analista de atendimento."
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": assistant_answer
            }
        ]
    }

In [ ]:
dataset_train = dataset_train.map(create_example)

In [ ]:
dataset_train[0]["messages"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.config.use_cache = False


In [ ]:
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES
)

In [ ]:
def formatting_func(example):

    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_train,
    peft_config=peft_config,
    formatting_func=formatting_func,
    args=training_args
)

In [ ]:
trainer.train()

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("Device:", next(model.parameters()).device)

for name, param in model.named_parameters():
    print(name, param.device)
    break

In [ ]:
trainer.save_model(OUTPUT_DIR)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)


In [ ]:
complaint = """
Meu cartão foi bloqueado sem aviso.

Tenho pagamentos vencendo e
não consigo atendimento.
"""

messages = [
    {
        "role": "system",
        "content":
        "Você é um analista de atendimento."
    },
    {
        "role": "user",
        "content": f"""
Analise a reclamação:

{complaint}

Retorne:

Categoria
Produto
Urgência
Problema Principal
Resposta Inicial
"""
    }
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(lora_model.device)

outputs = lora_model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)


In [ ]:
def generate(model, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
test_prompt = """
Cliente relata cobrança indevida
em cartão de crédito.
"""

In [ ]:
print("===== BASE =====")
print(generate(base_model, test_prompt))

In [ ]:
print("===== LoRA =====")
print(generate(lora_model, test_prompt))

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
print("Treinamento concluído.")
print("Modelo salvo em:", OUTPUT_DIR)